<!-- nav -->
[← 7 · From one language to another](07_From_One_Language_To_Another.ipynb) · [Home](../../README.md) · [9 · Constraints in hardware →](09_Constraints_In_Hardware.ipynb)

# 8 · From constraints to code

**The problem.** The cold chain's constraint, "the temperature stays between 2 and 8 °C", is kept as data with
mbse-expressions, so it's stored, checked and translated in one place. But the nightly pipeline needs it as a Python
function, written the way a Python programmer would write it, in a module that passes code review.

A **bridge** carries expressions between an mbse-expressions dialect and a language's syntax trees, both ways.
Constraints become code, generated rather than hand-copied, and existing code becomes constraints. That closes the loop
the mbse repositories are for: a specification that is executable, and the source code generated from it.

In [ ]:
from mbse.Expressions import Evaluators, Expressions as E, Translators
from mbse.Expressions.Dialects.Excel import Expressions as Excel, Text as ExcelText
from mbse.Expressions.Dialects.Python import Expressions as PythonDialect
from mbse.Programs.Bridges import Python as Bridge
from mbse.Programs.Framework import Errors
from mbse.Programs.Python import Python314, Syntax as P
from mbse.Schemas.Framework import Proxies, Schemas

Py = P.LANGUAGE.Builders
Shipment = (Schemas.OfObject.Builder().name("Shipment").ref()
            .properties(lambda p: p.name("celsius").of(lambda t: t.as_native(float)),
                        lambda p: p.name("hours").of(lambda t: t.as_native(int))).create())
store = Proxies.OfStore()
store.register(Shipment)
this = E.variable("this")
celsius = this.celsius
constraints = {
    "in_range": E.literal(2.0).le(celsius).and_(celsius.le(8.0)).data,
    "on_time": this.hours.le(72).data,
}

## Step 1: a constraint becomes a function

mbse-expressions translates the constraint from its neutral form (Basic) to its Python dialect; the bridge turns that
into a Python syntax tree, here a function of `this` that returns it:

In [ ]:
to_python = Translators.between(E.DIALECT, PythonDialect.DIALECT)
in_range = Bridge.function_("in_range", ["this"], to_python.forward(constraints["in_range"]))
print(Python314.print(in_range))

## Step 2: a module of checks

Every constraint becomes a function, and the functions a module, built as one tree and printed once:

In [ ]:
checks = Py.Module().body([Bridge.function_(name, ["this"], to_python.forward(constraint)) for name, constraint in constraints.items()])
source = Python314.print(checks.create())
print(source)

The module is ordinary Python, so it runs. Generated from the constraints, it agrees with them: each shipment below is
checked by the generated functions, then by evaluating the constraints themselves.

In [ ]:
namespace = {}
exec(source, namespace)
shipments = [store.Shipment().celsius(5.5).hours(30).create(), store.Shipment().celsius(9.1).hours(80).create()]
for shipment in shipments:
    generated = [namespace[name](shipment) for name in constraints]
    evaluated = [Evaluators.OfAny(constraint, {"this": shipment}) for constraint in constraints.values()]
    print(generated, evaluated)

## Step 3: code becomes a constraint

The bridge reads the other way too. A check already written in Python, in the pipeline nobody wants to rewrite, becomes
a constraint: stored as data, evaluated by Basic's rules, translated to Excel for the QA team.

In [ ]:
legacy = Python314.parse('''def fresh(this):
    return this.hours <= 48 and this.celsius >= 2.0
''')
term = Bridge.term(legacy.body[0].body[0].value)
constraint = Translators.between(PythonDialect.DIALECT, E.DIALECT).forward(term)
print(Evaluators.OfAny(constraint, {"this": shipments[0]}))
print(ExcelText.ToText(Translators.between(E.DIALECT, Excel.DIALECT).forward(constraint)))

## Step 4: what a constraint can't hold

A constraint is smaller than a program: pure, total, with no statements. Code outside that, here an identity test, can't
become one, and the bridge says where:

In [ ]:
try:
    Bridge.term(Python314.parse("this.celsius is None\n").body[0].value)
except Errors.TranspileError as error:
    print(error)

## Why it works this way

- **Specifications generate code.** A constraint written once, as data, is checked, stored and translated by
  mbse-expressions, and becomes production source here, printed by the language's own printer. Nobody copies it by
  hand, so nobody copies it wrong.
- **Both directions.** Existing code can be read into constraints, so formalizing a system can start from what's already
  written and proceed step by step.
- **Trees all the way.** From the constraint's terms to the dialect's terms to Python's syntax nodes, nothing pastes
  text, so precedence and quoting are right by construction.

That's the series: programs as data, changed safely, checked against standards, with names resolved, across languages,
shared between tools, translated between languages, and generated from the constraints of an executable specification.

<!-- nav -->
[← 7 · From one language to another](07_From_One_Language_To_Another.ipynb) · [Home](../../README.md) · [9 · Constraints in hardware →](09_Constraints_In_Hardware.ipynb)